In [3]:
import os
import glob
import os
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from tqdm import tqdm
import time
from scipy.spatial.distance import cdist
import scipy
from scanpy.tools._utils import get_init_pos_from_paga 

import rmm
import cupy
import cudf
import cupy as cp
# import cvxpy as cpv
from cuml.metrics import pairwise_distances
from cuml.metrics import nan_euclidean_distances
from rmm.allocators.cupy import rmm_cupy_allocator
import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi
import scipy.io
from scipy.sparse import load_npz, coo_matrix
from scipy.io import mmread
from scipy.sparse import csr_matrix

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

# Load in gene info

In [4]:
%%time
fpath = "../resources/gene_names.tsv.gz"
gdf = pd.read_csv(fpath, sep='\t', low_memory=False)
print(f"{gdf.shape=}")

protein_coding = gdf[gdf['Gene type'] == 'protein_coding']['Gene name'].unique()
print(f"N protein coding: {protein_coding.shape=}")

gdf.shape=(73466, 8)
N protein coding: protein_coding.shape=(18149,)
CPU times: user 126 ms, sys: 19.3 ms, total: 145 ms
Wall time: 173 ms


# Prepare Ng data

In [5]:
# tdata = sc.read_h5ad("/nfs/turbo/umms-indikar/shared/projects/HSC/data/datasets/ng_2024/iHSC.h5ad")
# tdata

AnnData object with n_obs × n_vars = 252607 × 27946
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'induction.ident', 'celltype', 'RETA'
    var: 'name'
    obsm: 'X_pca', 'X_umap'

In [5]:
# # Check if X is raw counts
# print("adata.raw is None?", tdata.raw is None)

# total_cells = tdata.n_obs

# # Randomly sample 100 unique indices
# sampled_indices = np.random.choice(total_cells, size=1000, replace=False)

# ddata = tdata[sampled_indices, :].copy()
# # tdata.X = tdata.layers['raw_counts'].copy()

# X = ddata.X.toarray() if hasattr(ddata.X, "toarray") else ddata.X
# print(X.dtype)
# print(np.min(X), np.max(X), np.mean(X))

adata.raw is None? True
float64
0.0 7.280478202167235 0.11040812909185703


## Load in raw counts (from R)

In [13]:
# fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_raw_counts.mtx"
# cells_path = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_cells.csv"
# genes_path = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_genes.csv"

# ng_counts = mmread(fpath).tocsr().transpose()
# ng_cells = pd.read_csv(cells_path, header=0).iloc[:,0].tolist()
# ng_genes = pd.read_csv(genes_path, header=0).iloc[:,0].tolist()

# ndata = an.AnnData(X=ng_counts)
# ndata.obs_names = ng_cells
# ndata.var_names = ng_genes

# ndata

AnnData object with n_obs × n_vars = 252607 × 27946

In [14]:
# # check if X is actually raw counts
# print("adata.raw is None?", ndata.raw is None)

# total_cells = ndata.n_obs

# # Randomly sample 100 unique indices
# sampled_indices = np.random.choice(total_cells, size=1000, replace=False)

# ddata = ndata[sampled_indices, :].copy()
# # tdata.X = tdata.layers['raw_counts'].copy()

# X = ddata.X.toarray() if hasattr(ddata.X, "toarray") else ddata.X
# print(X.dtype)
# print(np.min(X), np.max(X), np.mean(X))

adata.raw is None? True
int64
0 3048 0.38567580333500323


## Add raw counts to Ng data object

In [20]:
# # Check if names/orders match:
# print((tdata.obs_names == ndata.obs_names).all())  # Should be True
# print((tdata.var_names == ndata.var_names).all())  # Should be True

# tdata.layers['raw_counts'] = ndata.X.copy()

# print(tdata.layers['raw_counts'].shape)
# # Should be (252607, 27946)

# # Check a few values
# print(tdata.layers['raw_counts'][:5, :5])

# tdata

True
True
(252607, 27946)
<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 0 stored elements and shape (5, 5)>


AnnData object with n_obs × n_vars = 252607 × 27946
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'induction.ident', 'celltype', 'RETA'
    var: 'name'
    obsm: 'X_pca', 'X_umap'
    layers: 'raw_counts'

In [21]:
# print(tdata.layers['raw_counts'].nnz) # Number of nonzero values in entire matrix

774053504


In [22]:
# # write out
# outpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_2024_raw.h5ad"

# tdata.write(outpath)
# tdata

AnnData object with n_obs × n_vars = 252607 × 27946
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'induction.ident', 'celltype', 'RETA'
    var: 'name'
    obsm: 'X_pca', 'X_umap'
    layers: 'raw_counts'

In [ ]:
# file moved to Turbo: /nfs/turbo/umms-indikar/shared/projects/HSC/updated_adata/Ng_2024_raw.h5ad

# define datasets

In [23]:
datasets = {
    'Reference' :  {
        'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hematokytos/sample_1_adata.h5ad",
        'annotation_column' : 'cell_type',
        'layer' : 'counts',
        'filter_column' : 'basename',
        'filter_out' : ['myeloid_cells', 'lymphoid_cells', 'innate_lymphoid_cells'],
    },
    # 'Ng 2024' : {
    #     'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/data/datasets/ng_2024/iHSC.h5ad",
    #     'annotation_column' : 'celltype',
    #     'layer' : None,
    #     'filter_column' : None,
    #     'filter_out' : None,
    # },
    'Ng 2024' : {
        'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/updated_adata/Ng_2024_raw.h5ad",
        'annotation_column' : 'celltype',
        'layer' : 'raw_counts',
        'filter_column' : None,
        'filter_out' : None,
    },
    'Gomes 2018' : {
        'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/data/datasets/gomes_2018/gomes.h5ad",
        'annotation_column' : 'cell_type',
        'layer' : 'raw_counts',
        'filter_column' : None,
        'filter_out' : None,
    },
    'This Study' : {
        'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/updated_adata/three_libraries_new.h5ad",
        'annotation_column' : 'subgroup',
        'layer' : 'raw_counts',
        'filter_column' : None,
        'filter_out' : None,
    },
}

In [24]:
%%time
data = {}

for dataset, params in datasets.items():
    print(f"\n--- Loading dataset: {dataset} ---")
    t0 = time.time()

    fpath = params['file_path']
    column = params['annotation_column']
    layer = params['layer']
    filter_column = params['filter_column']
    filter_out = params['filter_out']

    print(f"\t - Reading file: {fpath}")
    bdata = sc.read_h5ad(fpath)
    bdata.var_names_make_unique()
    print(f"\t - Read complete. Shape: {bdata.shape}")
    print(f"\t - Obs columns: {', '.join(bdata.obs.columns)}")

    # Annotate cell type and filter
    bdata.obs['annotation'] = bdata.obs[column].copy()
    n_total = bdata.shape[0]
    bdata = bdata[bdata.obs['annotation'].notna(), :].copy()
    n_filtered = bdata.shape[0]
    print(f"\t - Cells before filtering: {n_total}, after filtering: {n_filtered}")

     # Select layer if specified
    if layer is not None:
        print(f"\t - Using layer: {layer}")
        bdata.X = bdata.layers[layer].copy()
    else:
        print("\t - Using .X as is (no layer specified)")

    
    print(f"\t - Filtering...")
    if filter_column is None:
        print(f"\t - No Defined filtering.")
    else:
        print(f"\t - Filtering {filter_out} from {filter_column}...")
        bdata = bdata[~bdata.obs[filter_column].isin(filter_out), :].copy()

    print("\t - Filtering non coding genes...")
    bdata = bdata[:, bdata.var_names.isin(protein_coding)].copy()
    
    bdata.X = bdata.X.astype('float32')
    rsc.get.anndata_to_GPU(bdata)
    rsc.pp.filter_genes(bdata, min_counts=10)
    
    print(f"\t - Final Shape: {bdata.shape}")

    n_genes = bdata.shape[1]
    n_cell_types = bdata.obs['annotation'].nunique()
    cell_types = sorted(bdata.obs['annotation'].unique())
    print(f"\t - Unique cell types ({n_cell_types}): {cell_types[:5]}{' ...' if n_cell_types > 5 else ''}")

    print(f"\t - Cleaning up metadata...")
    del bdata.obsm
    del bdata.obsp
    del bdata.layers
    del bdata.varm

    data[dataset] = bdata

    t1 = time.time()
    print(f"\t - Time to load and process: {(t1 - t0)/60:.2f} minutes")

print()
print(data.keys())
print('done.')


--- Loading dataset: Reference ---
	 - Reading file: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hematokytos/sample_1_adata.h5ad


/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


	 - Read complete. Shape: (1125041, 52164)
	 - Obs columns: soma_joinid, dataset_id, assay, assay_ontology_term_id, cell_type, cell_type_ontology_term_id, development_stage, development_stage_ontology_term_id, disease, disease_ontology_term_id, donor_id, is_primary_data, observation_joinid, self_reported_ethnicity, self_reported_ethnicity_ontology_term_id, sex, sex_ontology_term_id, suspension_type, tissue, tissue_ontology_term_id, tissue_type, tissue_general, tissue_general_ontology_term_id, raw_sum, nnz, raw_mean_nnz, raw_variance_nnz, n_measured_vars, basename, dataset_id_int, n_counts, n_genes, _scvi_batch, _scvi_labels, leiden


/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


	 - Cells before filtering: 1125041, after filtering: 1125041
	 - Using layer: counts
	 - Filtering...
	 - Filtering ['myeloid_cells', 'lymphoid_cells', 'innate_lymphoid_cells'] from basename...


/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


	 - Filtering non coding genes...
filtered out 295 genes that are detected in less than 10 counts
	 - Final Shape: (709808, 17524)
	 - Unique cell types (72): ['CD34-positive, CD38-negative hematopoietic stem cell', 'adipocyte', 'alveolar type 1 fibroblast cell', 'alveolar type 2 fibroblast cell', 'basophil mast progenitor cell'] ...
	 - Cleaning up metadata...
	 - Time to load and process: 5.65 minutes

--- Loading dataset: Ng 2024 ---
	 - Reading file: /scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_2024_raw.h5ad
	 - Read complete. Shape: (252607, 27946)
	 - Obs columns: orig.ident, nCount_RNA, nFeature_RNA, line.ident, percent.mt, percent.ribo, percent.mitoribo, S.Score, G2M.Score, Phase, integrated_snn_res.0.5, seurat_clusters, integrated_snn_res.1, integrated_snn_res.0.8, integrated_snn_res.0.3, integrated_snn_res.0.4, fract.ident, induction.ident, celltype, RETA
	 - Cells before filtering: 252607, after filtering: 252607
	 - Using layer: raw_counts
	 - Fil

/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


	 - Read complete. Shape: (286, 56269)
	 - Obs columns: cell_type, cell_num
	 - Cells before filtering: 286, after filtering: 286
	 - Using layer: raw_counts
	 - Filtering...
	 - No Defined filtering.
	 - Filtering non coding genes...
filtered out 3550 genes that are detected in less than 10 counts
	 - Final Shape: (286, 13492)
	 - Unique cell types (5): ['DAY15', 'DAY2', 'DAY25', 'HDF', 'UCB']
	 - Cleaning up metadata...
	 - Time to load and process: 0.02 minutes

--- Loading dataset: This Study ---
	 - Reading file: /scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/three_libraries_new.h5ad
	 - Read complete. Shape: (20754, 26933)
	 - Obs columns: dataset, source, cluster_all, group, label, S_score, G2M_score, phase, n_genes_by_counts, log1p_n_genes_by_counts, total_counts, log1p_total_counts, pct_counts_in_top_50_genes, pct_counts_in_top_100_genes, pct_counts_in_top_200_genes, pct_counts_in_top_500_genes, total_counts_mito, log1p_total_counts_mito, pct_counts_mito,

In [25]:
for k, v in data.items():
    print(k, v.shape)

Reference (709808, 17524)
Ng 2024 (252607, 15207)
Gomes 2018 (286, 13492)
This Study (20754, 15146)


In [26]:
for k, v in data.items():
    print('\n', k, list(v.obs.columns))


 Reference ['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden', 'annotation']

 Ng 2024 ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'inducti

In [27]:
%%time
max_cells = 250000
keep_columns = [
    'annotation',
    'cell_type',
    'celltype', 
    'tissue',
    'development_stage',
    'basename',
    'source', 
    'dataset_id',
]

adatas = []

print(f"Sampling up to {max_cells} cells per dataset...\n")

for key, bdata in data.items():
    print(f"Processing dataset: {key}")
    obs = bdata.obs.copy()
    
    # Keep only selected columns
    kept = [col for col in keep_columns if col in obs.columns]
    missing = [col for col in keep_columns if col not in obs.columns]
    print(f"  Keeping columns: {kept}")
    if missing:
        print(f"  Warning: missing columns: {missing}")
    
    obs = obs[kept]
    bdata.obs = obs
    bdata.obs['data_key'] = key
    bdata.obs['data_source'] = key

    # Sample cells
    n_cells = min(bdata.n_obs, max_cells)
    print(f"  Sampling {n_cells} cells (of {bdata.n_obs})")
    bdata = sc.pp.sample(bdata, n=n_cells, replace=False, copy=True)
    
    adatas.append(bdata)

print("\nConcatenating all datasets with outer join on genes...")
adata = an.concat(
    adatas, 
    join='outer',
    label='data_source',
    index_unique=None,
)

print(f"Final AnnData object: {adata.shape[0]} cells × {adata.shape[1]} genes")

adata

Sampling up to 250000 cells per dataset...

Processing dataset: Reference
  Keeping columns: ['annotation', 'cell_type', 'tissue', 'development_stage', 'basename', 'dataset_id']
  Sampling 250000 cells (of 709808)


/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Processing dataset: Ng 2024
  Keeping columns: ['annotation', 'celltype']
  Sampling 250000 cells (of 252607)
Processing dataset: Gomes 2018
  Keeping columns: ['annotation', 'cell_type']
  Sampling 286 cells (of 286)
Processing dataset: This Study
  Keeping columns: ['annotation', 'source']
  Sampling 20754 cells (of 20754)

Concatenating all datasets with outer join on genes...


/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/cupyx/scipy/sparse/_compressed.py:548: SparseEfficiencyWarning: Changing the sparsity structure of a csc_matrix is expensive.
  warnings.warn('Changing the sparsity structure of a '


Final AnnData object: 521040 cells × 17607 genes
CPU times: user 5min 38s, sys: 23.5 s, total: 6min 1s
Wall time: 9min 35s


<timed exec>:41: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 521040 × 17607
    obs: 'annotation', 'cell_type', 'tissue', 'development_stage', 'basename', 'dataset_id', 'data_key', 'data_source', 'celltype', 'source'

In [28]:
# check raw
# check if X is actually raw counts
print("adata.raw is None?", adata.raw is None)

total_cells = adata.n_obs

# Randomly sample 100 unique indices
sampled_indices = np.random.choice(total_cells, size=10000, replace=False)

ddata = adata[sampled_indices, :].copy()
# tdata.X = tdata.layers['raw_counts'].copy()

X = ddata.X.toarray() if hasattr(ddata.X, "toarray") else ddata.X
print(X.dtype)
print(np.min(X), np.max(X), np.mean(X))

adata.raw is None? True


/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


float64
nan nan nan


In [29]:
print(np.isnan(X).sum(), X.size)  # number of nan entries vs total
print(np.unique(np.isnan(X)))

12831057 176070000
[False  True]


In [30]:
adata.obs['data_key'].value_counts()

data_key
Reference     250000
Ng 2024       250000
This Study     20754
Gomes 2018       286
Name: count, dtype: int64

In [31]:
%%time
outpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/single_cell_atlas.h5ad"
adata.write(outpath)
adata

... storing 'data_key' as categorical


CPU times: user 9.1 s, sys: 11.5 s, total: 20.6 s
Wall time: 34.9 s


AnnData object with n_obs × n_vars = 521040 × 17607
    obs: 'annotation', 'cell_type', 'tissue', 'development_stage', 'basename', 'dataset_id', 'data_key', 'data_source', 'celltype', 'source'

In [ ]:
# file moved to Turbo: /nfs/turbo/umms-indikar/shared/projects/HSC/updated_adata/single_cell_atlas.h5ad